# Legal Proposition Generation with Saul-7B

This notebook uses the **Saul-7B-Instruct-v1** model to transform legal document fragments into atomic, retrieval-friendly legal propositions.

### Workflow:
1. Load extracted marked paragraphs (Parquet format).
2. Run batch inference using Saul-7B with 4-bit quantization.
3. Save generated propositions for use in the local retrieval pipeline.

In [ ]:
!pip install -q transformers accelerate bitsandbytes pandas tqdm pyarrow

In [ ]:
import torch
import pandas as pd
import json
import os
from transformers import pipeline, BitsAndBytesConfig, AutoTokenizer
from tqdm.auto import tqdm
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

## Configuration
Update the paths below to match your Google Drive structure.

In [ ]:
# Paths
INPUT_PARQUET = '/content/drive/MyDrive/legal-case-retrieval/data/test-files/processed/query_marked_paragraphs.parquet'
OUTPUT_PARQUET = '/content/drive/MyDrive/legal-case-retrieval/data/test-files/processed/query_propositions_saul.parquet'

# Model Configuration
MODEL_ID = "Equall/Saul-7B-Instruct-v1"
BATCH_SIZE = 8  # Adjust based on VRAM (L4 handles 8-16 well)

PROMPT_TEMPLATE = """### System: You are a legal expert. Convert the following legal fragment into a single, clear, and concise legal proposition. 
Keep it factual and neutral. Ensure the proposition is understandable in isolation.

### User: {fragment}

### Assistant:"""

## Initialize Model
Loading with 4-bit quantization for efficiency on L4/T4 GPUs.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

pipe = pipeline(
    "text-generation",
    model=MODEL_ID,
    model_kwargs={"quantization_config": bnb_config},
    device_map="auto",
    tokenizer=tokenizer
)

## Run Inference

In [ ]:
df = pd.read_parquet(INPUT_PARQUET)
print(f"Loaded {len(df)} paragraphs.")

results = []
fragments = df['cleaned_text'].tolist()
prompts = [PROMPT_TEMPLATE.format(fragment=f) for f in fragments]

for i in tqdm(range(0, len(prompts), BATCH_SIZE), desc="Generating Propositions"):
    batch_prompts = prompts[i : i + BATCH_SIZE]
    
    outputs = pipe(
        batch_prompts,
        max_new_tokens=256,
        do_sample=False, # Deterministic for propositions
        batch_size=BATCH_SIZE,
        return_full_text=False
    )
    
    for j, out in enumerate(outputs):
        idx = i + j
        gen_text = out[0]['generated_text'].strip()
        
        results.append({
            'doc_id': df.iloc[idx]['doc_id'],
            'paragraph_id': df.iloc[idx]['paragraph_id'],
            'cleaned_text': df.iloc[idx]['cleaned_text'],
            'proposition': gen_text
        })

# Save Results
df_propositions = pd.DataFrame(results)
df_propositions.to_parquet(OUTPUT_PARQUET, index=False)

print(f"Done! Propositions saved to {OUTPUT_PARQUET}")